# PostgreSQL Transactions and Lock Diagnosis

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/CST4714_OER_Rebuild/notebooks/02_postgres_transactions_locks.ipynb)

This notebook creates one controlled blocking relationship in a disposable schema,
identifies the blocked and blocking sessions, resolves the blocker, and verifies
the final row.

**The central idea:** a wait is a relationship between database sessions. Diagnose
that relationship before terminating or changing anything.

The cloud path asks for a temporary PostgreSQL URL with `getpass`, so the value is
not displayed or written into the notebook. The fallback path supplies a complete
incident transcript when a cloud connection is unavailable.

## Before You Connect

1. Use a personal course database, never a production system.
2. In Supabase, prefer the session-pooler URI if your network cannot reach the
   direct IPv6 endpoint.
3. Rotate the temporary password after class if required by your course policy.
4. Never paste a URL into a code or Markdown cell.

Set `USE_CLOUD` to `True` only when you are ready. It remains `False` in the public
notebook so all non-cloud cells can run safely without credentials.

In [1]:
%pip -q install "psycopg[binary]"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from getpass import getpass
import threading
import time

import psycopg

USE_CLOUD = False  # Change to True during the in-class cloud lab.
print("Cloud path enabled:", USE_CLOUD)

Cloud path enabled:

 False


## 1. Open Three Clearly Named Sessions

- **Session A** will update a row and deliberately remain uncommitted.
- **Session B** will attempt a competing update and wait.
- **Diagnostic session** will query PostgreSQL's activity evidence.

Three connections make the roles visible. The diagnostic session does not cause
or resolve the block; it observes it.

In [3]:
if USE_CLOUD:
    database_url = getpass("Paste the temporary PostgreSQL connection URL: ")

    session_a = psycopg.connect(database_url, application_name="cst4714_session_a")
    session_b = psycopg.connect(database_url, application_name="cst4714_session_b")
    diagnostic = psycopg.connect(database_url, application_name="cst4714_diagnostic")
    diagnostic.autocommit = True

    print("Opened Session A, Session B, and the diagnostic session.")
else:
    print("Cloud path skipped. Continue to the fallback transcript below.")

Cloud path skipped. Continue to the fallback transcript below.


## 2. Create a Disposable Target

The table has one row. Its original state is `priority = medium` and
`status = open`. Recreating the schema makes the exercise repeatable.

In [4]:
if USE_CLOUD:
    with diagnostic.cursor() as cursor:
        cursor.execute("DROP SCHEMA IF EXISTS lock_lab CASCADE")
        cursor.execute("CREATE SCHEMA lock_lab")
        cursor.execute("""
            CREATE TABLE lock_lab.tickets (
                ticket_id integer PRIMARY KEY,
                priority text NOT NULL,
                status text NOT NULL
            )
        """)
        cursor.execute("""
            INSERT INTO lock_lab.tickets (ticket_id, priority, status)
            VALUES (1004, 'medium', 'open')
        """)
        cursor.execute("SELECT * FROM lock_lab.tickets")
        print("Starting row:", cursor.fetchone())
else:
    print("Setup skipped because USE_CLOUD is False.")

Setup skipped because USE_CLOUD is False.


## 3. Session A Holds an Uncommitted Row Change

Session A changes the priority but does not commit. PostgreSQL holds the row-level
write conflict until the transaction ends.

In [5]:
if USE_CLOUD:
    with session_a.cursor() as cursor:
        cursor.execute("BEGIN")
        cursor.execute("""
            UPDATE lock_lab.tickets
            SET priority = 'high'
            WHERE ticket_id = 1004
            RETURNING ticket_id, priority, status
        """)
        print("Session A sees:", cursor.fetchone())
    print("Session A remains open and uncommitted.")
else:
    print("Session A step skipped.")

Session A step skipped.


## 4. Session B Starts a Competing Update

The notebook uses one small background thread because a blocked query cannot both
wait and let the same notebook cell continue to collect diagnostics. The thread
contains only Session B's query; all database actions remain visible below.

In [6]:
blocked_result = {}

def run_session_b_update():
    try:
        with session_b.cursor() as cursor:
            cursor.execute("SET statement_timeout = '15s'")
            cursor.execute("""
                UPDATE lock_lab.tickets
                SET status = 'in_progress'
                WHERE ticket_id = 1004
                RETURNING ticket_id, priority, status
            """)
            blocked_result["row"] = cursor.fetchone()
        session_b.commit()
        blocked_result["outcome"] = "committed after blocker released"
    except Exception as error:
        session_b.rollback()
        blocked_result["outcome"] = f"error: {type(error).__name__}: {error}"


if USE_CLOUD:
    session_b_thread = threading.Thread(target=run_session_b_update)
    session_b_thread.start()
    time.sleep(1)
    print("Session B update started. Thread still waiting:", session_b_thread.is_alive())
else:
    print("Session B step skipped.")

Session B step skipped.


## 5. Ask PostgreSQL Who Is Blocking Whom

`pg_blocking_pids(blocked.pid)` directly reports the blocker relationship. The
wait event adds context. A PID is evidence, not automatic permission to terminate
a process.

In [7]:
if USE_CLOUD:
    diagnostic_sql = """
        SELECT
            blocked.pid AS blocked_pid,
            blocked.application_name AS blocked_app,
            blocked.wait_event_type,
            blocked.wait_event,
            pg_blocking_pids(blocked.pid) AS blocking_pids,
            blocker.pid AS blocking_pid,
            blocker.application_name AS blocking_app,
            blocker.xact_start AS blocker_transaction_start,
            left(blocked.query, 90) AS blocked_query
        FROM pg_stat_activity AS blocked
        CROSS JOIN LATERAL unnest(pg_blocking_pids(blocked.pid)) AS p(blocking_pid)
        JOIN pg_stat_activity AS blocker
          ON blocker.pid = p.blocking_pid
        WHERE blocked.datname = current_database()
          AND blocked.application_name = 'cst4714_session_b'
    """
    with diagnostic.cursor() as cursor:
        cursor.execute(diagnostic_sql)
        columns = [description.name for description in cursor.description]
        rows = cursor.fetchall()
    print(columns)
    for row in rows:
        print(row)
else:
    print("Diagnostic query skipped.")

Diagnostic query skipped.


## 6. Resolve the Blocker and Verify the Final State

This controlled exercise rolls back Session A. That releases its row lock without
keeping the priority change. Session B can then finish and commit its status
change. A fresh diagnostic query verifies which values remain.

In [8]:
if USE_CLOUD:
    session_a.rollback()
    print("Rolled back Session A.")

    session_b_thread.join(timeout=20)
    print("Session B outcome:", blocked_result)

    with diagnostic.cursor() as cursor:
        cursor.execute("SELECT ticket_id, priority, status FROM lock_lab.tickets")
        final_row = cursor.fetchone()
    print("Final row from a fresh statement:", final_row)
else:
    print("Resolution step skipped.")

Resolution step skipped.


## 7. Close Connections and Remove the Disposable Schema

Cleanup is part of the operation. It prevents an old practice lock or test table
from becoming a later mystery.

In [9]:
if USE_CLOUD:
    with diagnostic.cursor() as cursor:
        cursor.execute("DROP SCHEMA IF EXISTS lock_lab CASCADE")

    session_a.close()
    session_b.close()
    diagnostic.close()
    database_url = None
    print("Closed all sessions and removed lock_lab.")
else:
    print("No cloud resources were opened.")

No cloud resources were opened.


## Offline Fallback Incident Transcript

Use this transcript when the cloud path cannot run. It represents the same
controlled experiment:

```text
Starting row: (1004, 'medium', 'open')

Session A PID 7310, application cst4714_session_a
Transaction began: 2026-03-05 18:42:10+00
Uncommitted query: UPDATE lock_lab.tickets SET priority = 'high'
                   WHERE ticket_id = 1004

Session B PID 7332, application cst4714_session_b
State: active
wait_event_type: Lock
wait_event: transactionid
pg_blocking_pids(7332): {7310}
Query: UPDATE lock_lab.tickets SET status = 'in_progress'
       WHERE ticket_id = 1004

Action: ROLLBACK issued in Session A.
Session B outcome: committed after blocker released.
Final row: (1004, 'medium', 'in_progress')
```

The transcript proves one blocking relationship, the chosen resolution, and the
final state. It does not prove how a production application should choose between
waiting, rollback, cancellation, or termination.

## Incident Record: Complete Before Submission

Edit this cell with your own cloud evidence or the fallback transcript.

**Blocked session:** [PID, application name, wait event, and query]

**Blocking session:** [PID, application name, and transaction start]

**Relationship evidence:** [the exact `pg_blocking_pids` result]

**Resolution:** [what ended the blocking transaction and why that action was safe
in this lab]

**Final verification:** [the final row and which change remained]

**Limitation:** [one thing this controlled test does not prove about production]

**Credential check:** I confirm that no connection URL, password, or key appears
in this notebook source or output. [replace with yes after checking]

**License:** prose CC BY-NC-SA 4.0; code MIT.